# Walmart Weekly Sales Prediction

## Business Problem

Accurate sales forecasting can support inventory planning, resource allocation, and business decision-making. This project aims to predict weekly sales across Walmart stores using historical sales data and several economic and temporal features.

## Project Objective

The main objective is to build and evaluate machine learning regression models capable of predicting `weekly_sales` while respecting the chronological nature of the dataset.

Several regression algorithms are compared:

- Linear Regression
- Polynomial Regression with Ridge Regularization
- K-Nearest Neighbors (KNN)
- Decision Tree Regressor
- Random Forest Regressor

Model selection is performed using a validation set and time-aware cross-validation. The final selected model is then retrained on the combined training and validation data and evaluated once on a held-out future test set.

## Evaluation Metrics

Model performance is evaluated using:

- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Squared Error)**
- **R² (Coefficient of Determination)**

# **Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use(style='fivethirtyeight')


# **Load Data**

In [ ]:
path = r'Walmart.csv'
df = pd.read_csv(path)
df.head()

# **Data Overview**

In [ ]:
df.info()

In [ ]:
# Change float format in data
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
df.describe().round(2)

## Data Preparation and Preprocessing

In this stage, the dataset is prepared for machine learning by cleaning the data, engineering relevant temporal features, handling selected outliers, and creating a chronological train-validation-test split.

The preprocessing steps are designed to avoid data leakage by fitting transformations only on the training data through a Scikit-learn pipeline.

### Rename Columns

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.rename({'holiday_flag': 'is_holiday'}, axis=1, inplace=True)
df.columns

### Missing Data

In [ ]:
df.isnull().sum()

* No missing values in the dataset

### Duplicate Values

In [ ]:
df.duplicated().sum()

* No duplicate values in the dataset

### Data Type

In [ ]:
df['date'] = pd.to_datetime(df['date'], format = "%d-%m-%Y")

## Feature Engineering
Temporal features were extracted from the `date` column to support exploratory analysis and model development.

- `year`, `quarter`, `season`, and `month_name` were mainly used for exploratory analysis.
- `month` and `week` were retained as temporal predictors for model training.
- The original `date` column was preserved for chronological splitting but was excluded from the model input features.

In [ ]:
# Create a new column "year" containing the year
df['year'] = df['date'].dt.year

# Create a new column "quarter" containing the season number
df['quarter'] = df['date'].dt.quarter

In [ ]:
# For this analysis, seasons are defined using calendar quarters:
# Q1 = Winter, Q2 = Spring, Q3 = Summer, and Q4 = Fall.

def get_season(quarter):
    "Find season based on quarter number"
    if quarter == 1: 
        return 'Winter'
    elif quarter == 2:
        return 'Spring'
    elif quarter == 3:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['quarter'].apply(get_season)

In [ ]:
# Create a new column "month" containing the month number
df['month'] = df['date'].dt.month

# Create a new column "month_name" containing the month names
df['month_name'] = df['date'].dt.month_name()

# Create a new column "week" containing the week number
df['week'] = df['date'].dt.isocalendar().week

In [ ]:
df['week'] = df['week'].astype('int32')

In [ ]:
df.head()

# **Exploratory Data Analysis (EDA)**

## Univariate Analysis

### Numerical Features

In [ ]:
num_cols = ['weekly_sales', 'temperature', 'fuel_price', 'unemployment', 'cpi']

plt.figure(figsize=(18, 20))
for i, col in enumerate(num_cols):
    plt.subplot(3,2, i+1)
    sns.histplot(data= df, x = col, kde=True, bins= 15, color='r')

plt.show()

### Distribution Analysis Findings

- `weekly_sales` has a right-skewed distribution, indicating that some store-week observations have substantially higher sales than others.
- `temperature` and `unemployment` show relatively concentrated distributions, but their normality cannot be established from histograms alone.
- `cpi` and `fuel_price` show multiple concentrations in their distributions, which may reflect differences across stores, regions, or time periods.

These observations are descriptive and do not establish causal relationships with weekly sales.

### Categorical Features

In [ ]:
# is_holiday
df['is_holiday'].value_counts()

In [ ]:
sns.countplot(data =df, x= 'is_holiday', palette='Set2');

In [ ]:
plt.pie(df['is_holiday'].value_counts().values, labels = ['Not Holidays', 'Holidays'], autopct='%1.2f%%');

* Most observations represent non-holiday weeks, while holiday weeks
account for a smaller proportion of the dataset.

In [ ]:
# year
fig, ax = plt.subplots(1, 2, figsize = (14, 6))
sns.countplot(data = df, x = 'year', ax = ax[0])
ax[1].pie(df['year'].value_counts().values,
          labels = df['year'].value_counts().index,
          autopct = '%1.1f%%')
plt.show()

- The number of observations differs across years because the dataset covers only part of 2010 and 2012, while 2011 contains a full year of weekly observations. Therefore, the higher frequency of observations in 2011 should not be interpreted as higher sales performance.

In [ ]:
# season
df['season'].value_counts()

In [ ]:
sns.countplot(data = df, x = 'season', palette='Set2');


- Summer has the highest number of observations in the dataset. However, this reflects the distribution of recorded weeks across the available time period and should not be interpreted as evidence that summer has higher weekly sales.

In [ ]:
# Month name
df['month_name'].value_counts()

In [ ]:
plt.figure(figsize=(16, 6));
sns.countplot(data = df, x = 'month_name', palette= 'Set2');
plt.xlabel('Month');
plt.show();

- April and July contain the highest number of observations in the dataset. These differences reflect the dataset's time coverage and should not be interpreted as differences in monthly sales performance.

In [ ]:
plt.figure(figsize=(15, 7))
sns.countplot(data = df, x = 'store')


- All 45 stores contain the same number of observations in the original dataset, indicating balanced store-level coverage before outlier filtering.

## Bi-variate Analysis

**Weekely_sales & is_holiday**

In [ ]:
df.groupby('is_holiday')['weekly_sales'].mean()

In [ ]:
plt.figure(figsize = (12, 6))
sns.barplot(data = df, x = 'is_holiday', y = 'weekly_sales',
            estimator = np.mean, ci = False, palette='Set2')

# Add labels and title
plt.title('Average Sales by Holidays')
plt.xlabel('Is Holiday', size = 12)
plt.ylabel('Average Sales', size = 12)
plt.show()

* Average weekly sales are higher during holiday weeks than during
non-holiday weeks in the observed dataset. This is an association
and does not establish that holidays alone caused the increase.

**weekly_sales & store**

In [ ]:
gb_store = df.groupby('store')['weekly_sales'].sum().sort_values(ascending = False)
gb_store

In [ ]:
plt.figure(figsize=(18, 6))
sns.barplot( x = gb_store.index, y = gb_store.values, color = 'steelblue')

plt.title('Total Sales by Store')
plt.xlabel('Store')
plt.ylabel('Total Sales')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

- There is a high variance in weekly sales from one store to another.
- Store No. 20 has the highest sales from any store followed by Store No. 4 and Store No. 33 comes last.

**weekly_sales & temperature**

In [ ]:
plt.figure(figsize = (14, 5))
sns.scatterplot(data = df, x = 'temperature', y = 'weekly_sales', edgecolor = "black")

# Add labels and title
plt.title('Sales by Temperature', size = 20)
plt.xlabel('Temperature', size = 15)
plt.ylabel('Sales', size = 15)
plt.show()

**weekly_sales & fule_price**

In [ ]:
plt.figure(figsize = (14, 5))
sns.scatterplot(data = df, x = 'fuel_price', y = 'weekly_sales', color = 'black', edgecolor = "red")

# Add labels and title
plt.title('Sales by Fuel Price', size = 20)
plt.xlabel('Fuel Price', size = 15)
plt.ylabel('Sales', size = 15)
plt.show()

- The scatter plots do not show a clear overall linear relationship
between weekly sales and temperature or fuel price.

However, the relationships may differ across stores and seasons.
These visualizations alone are insufficient to conclude that
temperature and fuel price have no effect on sales.

**weekly_sales & cpi**

In [ ]:
plt.figure(figsize = (14, 5))
sns.scatterplot(data = df, x = 'cpi', y = 'weekly_sales', color = '#8de5a1', edgecolor = "black")

# Add labels and title
plt.title('Sales by Consumer Price Index', size = 20)
plt.xlabel('CPI', size = 15)
plt.ylabel('Sales', size = 15)
plt.show()

The scatter plot does not reveal a clear overall linear relationship
between CPI and weekly sales.

The concentration of observations within different CPI ranges may
reflect regional or temporal differences. Customer income groups
cannot be inferred from CPI values in this dataset.

**weekly_sales & unemployment**

In [ ]:
plt.figure(figsize = (14, 5))
sns.scatterplot(data = df, x = 'unemployment', y = 'weekly_sales', color = 'blue', edgecolor = 'black')

# Add labels and title
plt.title('Sales by Unemployment Rate', size = 20)
plt.xlabel('Unemployment Rate', size = 15)
plt.ylabel('Sales', size = 15)
plt.show()

The scatter plot suggests a possible negative association between
unemployment and weekly sales in parts of the data.

However, the overall relationship is not sufficient to establish
a causal effect, and differences between stores may influence
the observed pattern.

### Time Series Analysis

* **date**

In [ ]:
weekly_total = (df.groupby('date')['weekly_sales'].sum().sort_index())

plt.figure(figsize=(20, 6))
plt.plot(weekly_total.index, weekly_total.values)

plt.title('Total Weekly Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Weekly Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

* **year**

In [ ]:
df.groupby('year')['weekly_sales'].sum().sort_values(ascending=False)

In [ ]:
plt.figure(figsize = (14, 6))
sns.barplot(data = df, x = 'year', y = 'weekly_sales', estimator = np.sum,
            ci = False,  palette= 'Set2')

# Add labels and title
plt.title('Total Sales in each Year')
plt.xlabel('Year', size = 13)
plt.ylabel('Total Sales', size = 13)
plt.show()

Annual sales totals are affected by the number of weeks available
in each year. Therefore, average total weekly sales are also
compared to provide a more consistent year-level view.

* **season**

In [ ]:
df.groupby('season')['weekly_sales'].sum().sort_values(ascending = False)

In [ ]:
plt.figure(figsize = (14, 6))
sns.barplot(data = df, x = 'season', y = 'weekly_sales', estimator = np.sum,
            ci = False, palette='Set2')

# Add labels and title
plt.title('Total Sales in each Season')
plt.xlabel('Season', size = 13)
plt.ylabel('Total Sales', size = 13)
plt.show()

Summer recorded the highest aggregated sales across the observation
period, totaling approximately $1.842 billion.

However, seasonal totals may be influenced by differences in the
number of weeks represented in each season.

* **month**

In [ ]:
monthly_sales = (df.groupby('month')['weekly_sales'].sum().reindex(range(1, 13)))


In [ ]:
month_labels = [
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(14, 5))
plt.plot(month_labels, monthly_sales.values, marker='o')

plt.title('Total Sales by Calendar Month')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()

* **week**

In [ ]:
gp_week = df.groupby('week')['weekly_sales'].sum().sort_values(ascending = False)
gp_week

In [ ]:
plt.figure(figsize = (18, 5))
sns.lineplot(data = df, x = 'week', y = 'weekly_sales', estimator = np.sum)

# Add labels and title
plt.title('Total Sales in each Week')
plt.xlabel('Week', size = 13)
plt.ylabel('Total Sales', size = 13)
plt.show()

- Total sales for all years in week 51 are the highest from any week, with 157,929,657$.

* **total sales in each year regarding the month**

In [ ]:
pd.pivot_table(data = df, index = 'year', columns = 'month_name', values = 'weekly_sales', aggfunc = 'sum')

In [ ]:
plt.figure(figsize = (16, 6))
sns.barplot(data = df, x = 'year', y = 'weekly_sales', hue = 'month',
            estimator = np.sum, ci = False)

# Add labels and title
plt.title('Total Sales for each Month in each Year')
plt.xlabel('Year', size = 18)
plt.ylabel('Total Sales', size = 18)

plt.show()

December recorded the highest monthly sales in both 2010 and 2011:

- December 2010: approximately $288.76 million.
- December 2011: approximately $288.08 million.

The 2012 dataset ends in October, so its final months cannot be
compared directly with the complete year of 2011.

the total sales in each year regarding the season

In [ ]:
pd.pivot_table(data = df, index = 'year', columns = 'season', values = 'weekly_sales', aggfunc = 'sum')

In [ ]:
plt.figure(figsize = (16, 6))
sns.barplot(data = df, x = 'year', y = 'weekly_sales',
            hue = 'season', estimator = np.sum, ci = False)

# Add labels and title
plt.title('Total Sales for each Season in each Year')
plt.xlabel('Year', size = 15)
plt.ylabel('Total Sales', size = 15)
plt.show()

Total sales in Autumn 2010 were the highest in the three years, where:

   - In 2010, total sales in Fall were the highest with 708,775,727.
   - In 2011, total sales in Fall were the highest with 681,501,740.
   - In 2012, total sales in Spring were the highest with 618,297,715$.

In [ ]:
years = ['2010', '2011', '2012']
colors = ['red', 'black', 'blue']

plt.figure(figsize = (18, 5))
for i, year in enumerate(years):
    sns.lineplot(data = df[df['year'] == int(year)],
                 x = 'month',
                 y = 'weekly_sales',
                 estimator = np.sum,
                 color = colors[i],
                 label = year)

# Add labels and title
plt.title(f'Total Sales Over Time', size = 22)
plt.xlabel('Month', size = 20)
plt.ylabel('Total Sales', size = 20)

# Add a legend
plt.legend()

# Show the plot
plt.show()

Monthly sales show noticeable peaks toward the end of 2010 and 2011.
A direct comparison with the end of 2012 is not possible because
the dataset ends in October 2012.

In [ ]:
#### unemployment rate over time
plt.figure(figsize = (18, 5))
for i, year in enumerate(years):
    sns.lineplot(data = df[df['year'] == int(year)],
                 x = 'month',
                 y = 'unemployment',
                 estimator = np.mean,
                 color = colors[i],
                 label = year)

# Add labels and title
plt.title(f'Unemployment Rate Over Time', size = 22)
plt.xlabel('Date', size = 20)
plt.ylabel('Unemployment Rate', size = 20)

# Add a legend
plt.legend()

# Show the plot
plt.show()

- The unemployment rate decreases over time.

**fual price rate over time**

In [ ]:
plt.figure(figsize = (18, 5))
for i, year in enumerate(years):
    sns.lineplot(data = df[df['year'] == int(year)],
                 x = 'month',
                 y = 'fuel_price',
                 estimator = np.mean,
                 color = colors[i],
                 label = year)

# Add labels and title
plt.title(f'Fuel Price Over Time', size = 22)
plt.xlabel('Date', size = 20)
plt.ylabel('Fuel Price', size = 20)

# Add a legend
plt.legend()

# Show the plot
plt.show()

- The fuel price increases over time.

## Data Correlation

In [ ]:
# Correlations with weekly sales
corr_cols = ['weekly_sales', 'temperature', 'fuel_price', 'cpi', 'unemployment', 'is_holiday']

corr_matrix = df[corr_cols].corr()

display(
    corr_matrix[['weekly_sales']]
    .sort_values(by='weekly_sales', ascending=False)
    .style.background_gradient(cmap='RdYlBu')
)

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)

plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

### Correlation Findings

The correlation analysis examines overall linear associations
between weekly sales and the available numerical predictors,
including the binary holiday indicator.

Weak correlations do not necessarily imply that a feature is
irrelevant for prediction. Relationships may be nonlinear or
vary across stores and time periods.

## EDA Summary

* The exploratory analysis revealed several important characteristics of the Walmart sales dataset.

* The dataset contains weekly sales observations across 45 stores from February 2010 to October 2012. No missing values or duplicate observations were identified.

* Total sales differ substantially across stores, while holiday weeks show higher average sales than non-holiday weeks. The analysis also revealed seasonal patterns and sales peaks during certain periods.

* The economic variables showed limited overall linear relationships with weekly sales. However, these relationships may differ between stores and across time.

* Because the dataset contains repeated weekly observations for each store, its temporal structure must be considered during preprocessing, model development, and evaluation.

### Implications for Modeling

The next stage will examine feature representation, scaling, outliers, and an appropriate data-splitting strategy. Particular attention will be given to preventing information from future weeks from leaking into model training.

# Data Pre-processing

In [ ]:
df_copy = df.copy()

### Feature Selection for Modeling

A copy of the dataset was created for model development.

Features used only for exploratory analysis (`year`, `quarter`, `season`, and `month_name`) were removed from the modeling dataset. `day_of_week` was also excluded because all observations occur on Friday, making it a constant feature.

The `month` and `week` variables were retained as temporal predictors, while `date` was preserved temporarily to perform chronological data splitting and was excluded from the model input afterward.

In [ ]:
df_copy = df_copy.drop(columns=['year', 'quarter', 'season', 'month_name', 'day_of_week'], errors='ignore')

print(df_copy.columns.tolist())

In [ ]:
print("Shape:", df_copy.shape)
print("Stores:", df_copy['store'].nunique())

### **Outlier Handling**

Potential extreme values were identified during exploratory data analysis. For this experiment, observations meeting the following conditions were removed before model training:

- `temperature < 7`
- `unemployment < 4.4`
- `unemployment > 11`

This filtering decision was retained as part of the experimental preprocessing strategy. However, later error analysis showed that filtering removed all training and validation observations for some stores, creating unseen store categories in the final test set.

This consequence is documented and discussed in the **Limitations** section rather than modifying the preprocessing strategy after observing the test results.

In [ ]:
cols = ['fuel_price', 'temperature', 'cpi', 'unemployment']
plt.figure(figsize=(20,18))
for i,col in enumerate(cols):
    print(i, col)
    plt.subplot(3,2,i+1)
    sns.boxplot(df_copy, x = col, color = 'red')
plt.show()

print('Number of data rows: ', df_copy.shape[0])

In [ ]:
# Remove outliers from the temperature column
df_copy.drop(df_copy[df_copy['temperature'] < 7].index, axis = 0, inplace = True)

In [ ]:
# Remove outliers from the unemployment column
df_copy.drop(df_copy[df_copy['unemployment'] < 4.4].index, axis = 0, inplace = True)
df_copy.drop(df_copy[df_copy['unemployment'] > 11].index, axis = 0, inplace = True)

In [ ]:
cols = ['temperature', 'fuel_price', 'cpi', 'unemployment']
plt.figure(figsize=(20,18))
for i, col in enumerate(cols):
    print(i, col)
    plt.subplot(3,2,i+1)
    sns.boxplot(df_copy, x = col, color = 'g')
plt.show()

print('Number of data rows: ', df_copy.shape[0])

### Chronological Train-Validation-Test Split

Because the dataset represents weekly observations over time, a random split was avoided. Instead, the data was split chronologically using unique weekly dates so that all stores belonging to the same week remain in the same subset.

- **Training:** approximately 64% of unique weeks
- **Validation:** approximately 16%
- **Test:** approximately 20%

The test set represents the latest period and is reserved exclusively for final model evaluation.

# Splitting Data

In [ ]:
# Sort observations chronologically
df_copy = df_copy.sort_values(['date', 'store']).reset_index(drop=True)

# Split by unique weeks, not individual rows
unique_dates = np.sort(df_copy['date'].unique())

n_dates = len(unique_dates)

train_end = int(n_dates * 0.64)
val_end = int(n_dates * 0.80)

train_dates = unique_dates[:train_end]
val_dates = unique_dates[train_end:val_end]
test_dates = unique_dates[val_end:]

train_df = df_copy[df_copy['date'].isin(train_dates)].copy()
val_df = df_copy[df_copy['date'].isin(val_dates)].copy()
test_df = df_copy[df_copy['date'].isin(test_dates)].copy()

In [ ]:
# Split Features and Target
target = 'weekly_sales'

X_train = train_df.drop(columns=[target, 'date'])
y_train = train_df[target]

X_val = val_df.drop(columns=[target, 'date'])
y_val = val_df[target]

X_test = test_df.drop(columns=[target, 'date'])
y_test = test_df[target]

In [ ]:
for name, subset in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f"{name}:")
    print("Shape:", subset.shape)
    print("From:", subset['date'].min().date())
    print("To:", subset['date'].max().date())
    print()

In [ ]:
assert train_df['date'].max() < val_df['date'].min()
assert val_df['date'].max() < test_df['date'].min()

print("Temporal split verified.")

## **Data Transformation**

In [ ]:
print(X_train.dtypes)
print("\nMissing values:")
print(X_train.isnull().sum())

### Preprocessing Pipeline

Different preprocessing strategies were applied according to feature type:

- Numerical features (`temperature`, `fuel_price`, `cpi`, `unemployment`, `month`, and `week`) were median-imputed and standardized.
- `store` was treated as a categorical feature and encoded using One-Hot Encoding.
- `is_holiday` was retained as a binary feature without scaling.

`OneHotEncoder(handle_unknown='ignore')` was used so that previously unseen store categories would not cause prediction errors.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = ['temperature', 'fuel_price', 'cpi', 'unemployment', 'month', 'week']

categorical_features = ['store']
binary_features = ['is_holiday']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
    ('binary', 'passthrough', binary_features)
])

# Modeling

In [ ]:
# Model Evaluation Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def model_evaluation(estimator, Training_Testing, X, y):

    ''' This function is used to evaluate the model through RMSE and R2'''

    predictions = estimator.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    print(f"\n{Training_Testing} Results")
    print(f"MAE:  {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R²:   {r2:.4f}")

    return {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

In [ ]:
def Distribution_Plot(estimator, Training_Testing, X, y, title):
    y_pred = estimator.predict(X)

    plt.figure(figsize=(12, 5))

    sns.kdeplot(x=y, color='blue', label=f'Actual ({Training_Testing})')

    sns.kdeplot(x=y_pred, color='red', label=f'Predicted ({Training_Testing})')

    plt.title(title)
    plt.xlabel('Weekly Sales ($)')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    plt.show()

### Time-Aware Cross-Validation

Hyperparameter tuning was performed using time-aware cross-validation based on unique weekly dates. This preserves temporal order and prevents observations from future weeks from being used to train models evaluated on earlier weeks.

In [ ]:
# Cross Validation
from sklearn.model_selection import TimeSeriesSplit


# Sort training data chronologically
train_df = train_df.sort_values(['date', 'store']).reset_index(drop=True)

# Recreate X_train and y_train in the same order
X_train = train_df.drop(columns=['date', 'weekly_sales'])
y_train = train_df['weekly_sales']

# Create folds based on unique dates
unique_dates = np.sort(train_df['date'].unique())
tscv = TimeSeriesSplit(n_splits=5)

time_folds = []

for train_idx, val_idx in tscv.split(unique_dates):
    fold_train_dates = unique_dates[train_idx]
    fold_val_dates = unique_dates[val_idx]

    train_rows = np.flatnonzero(train_df['date'].isin(fold_train_dates))

    val_rows = np.flatnonzero(train_df['date'].isin(fold_val_dates))

    time_folds.append((train_rows, val_rows))

print("Number of folds:", len(time_folds))

In [ ]:
from sklearn.model_selection import cross_val_score

def cross_validation_score(estimator, X_train, y_train,score='r2', cv=time_folds
):
    scores = cross_val_score(estimator, X_train, y_train, scoring=score, cv=cv)

    print("Cross-validation scores:", scores)
    print(f"Mean R²: {scores.mean():.4f}")
    print(f"Standard deviation: {scores.std():.4f}")

    return scores

In [ ]:
# Hyperparameter Tunning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GridSearchCV

def hyperparameter_tunning(estimator, X_train, y_train, param_grid, score='neg_root_mean_squared_error', cv=None
):
    grid_search = GridSearchCV(estimator=estimator, param_grid=param_grid, scoring=score, cv=cv, n_jobs=-1, refit=True)

    grid_search.fit(X_train, y_train)

    print("Best parameters:", grid_search.best_params_)
    print(f"Best CV RMSE: {-grid_search.best_score_:,.2f}")

    return grid_search.best_estimator_

# **Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr.fit(X_train, y_train)

In [ ]:
# Training Evaluation
lr_train_results = model_evaluation(lr, 'Training', X_train, y_train)

# Validation Evaluation
lr_val_results = model_evaluation(lr, 'Validation', X_val, y_val)

In [ ]:
Distribution_Plot(lr, 'Validation', X_val, y_val, 'Linear Regression: Actual vs Predicted Sales')

# Polynomial Regression with Ridge Regularization

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ('model', Ridge(alpha=100.0))
])

poly_pipeline.fit(X_train, y_train)

In [ ]:
# Training and Validation Evaluation Before Tuning

poly_train_results = model_evaluation(poly_pipeline, 'Training', X_train, y_train)

poly_val_results = model_evaluation(
poly_pipeline, 'Validation', X_val, y_val)

In [ ]:
Title = 'Distribution Plot of Predicted Value Using Training Data vs Training Data Distribution'
Distribution_Plot(poly_pipeline, 'Training', X_train, y_train, Title)

In [ ]:
param_grid = {
    'model__alpha': [0.1, 1, 10, 100, 1000]
}

best_poly = hyperparameter_tunning(
    estimator=poly_pipeline,
    X_train=X_train,
    y_train=y_train,
    param_grid=param_grid,
    cv=time_folds
)

In [ ]:
best_poly_val_results = model_evaluation(
    best_poly,
    'Validation',
    X_val,
    y_val
)

# KNN Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

knn_pipeline.fit(X_train, y_train)

In [ ]:
knn_train_results = model_evaluation(knn_pipeline, 'Training', X_train, y_train)

knn_val_results = model_evaluation(knn_pipeline, 'Validation', X_val, y_val)

Tuning the Model

In [ ]:
knn_param_grid = {'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21],'model__weights': ['uniform', 'distance'],'model__p': [1, 2]
}

best_knn = hyperparameter_tunning(
    estimator=knn_pipeline,
    X_train=X_train,
    y_train=y_train,
    param_grid=knn_param_grid,
    cv=time_folds
)

In [ ]:
best_knn_val_results = model_evaluation(best_knn, 'Validation', X_val, y_val)

# **Decision Tree Regressor**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

tree_pipeline.fit(X_train, y_train)

In [ ]:
dt_train_results = model_evaluation(tree_pipeline, 'Training', X_train, y_train)

dt_val_results = model_evaluation(tree_pipeline, 'Validation', X_val, y_val)

The default Decision Tree achieved nearly perfect performance on the training data but substantially lower performance on the validation set, indicating overfitting. Hyperparameter tuning was therefore applied to improve generalization.

**Decision Tree Tuning**

The model has overfitting, it needs improving it by tuning the hyperparameters.

In [ ]:
dt_param_grid = {
    'model__max_depth': [3, 5, 7, 10, 15, None],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10]
}

best_dt = hyperparameter_tunning(estimator=tree_pipeline, X_train=X_train, y_train=y_train, param_grid=dt_param_grid, cv=time_folds)

In [ ]:
best_dt_val_results = model_evaluation(best_dt, 'Validation', X_val, y_val)

# **Random Forest Regressor (Bagging)**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)

In [ ]:
rf_train_results = model_evaluation(rf_pipeline, 'Training', X_train, y_train)

rf_val_results = model_evaluation(rf_pipeline, 'Validation', X_val, y_val)

**Random Forest Hyperparameter Tuning**

In [ ]:
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5],
    'model__max_features': ['sqrt', 0.7, 1.0]
}

best_rf = hyperparameter_tunning(estimator=rf_pipeline, X_train=X_train, y_train=y_train, param_grid=rf_param_grid, cv=time_folds)

In [ ]:
best_rf_val_results = model_evaluation(best_rf, 'Validation', X_val, y_val)

Although hyperparameter tuning improved the cross-validation objective, the tuned Random Forest performed worse than the default Random Forest on the held-out validation set. Therefore, both versions were retained in the final model comparison.

# Final Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Linear Regression', **lr_val_results},

    {'Model': 'Polynomial + Ridge (Tuned)', **best_poly_val_results},

    { 'Model': 'KNN (Tuned)', **best_knn_val_results}, 

    { 'Model': 'Decision Tree (Tuned)', **best_dt_val_results},

    { 'Model': 'Random Forest (Default)', **rf_val_results},

    { 'Model': 'Random Forest (Tuned)', **best_rf_val_results}
])

comparison = comparison.sort_values(by='RMSE').reset_index(drop=True)

comparison

### Model Selection

RMSE was used as the primary model-selection metric. The tuned KNN achieved the lowest validation RMSE (180,165.53), narrowly outperforming the default Random Forest (181,645.85).

Therefore, the tuned KNN was selected as the final model with:

- `n_neighbors = 3`
- `weights = 'distance'`
- `p = 1`

## Final Model Training

After model selection was completed using the validation set, the selected KNN configuration was retrained using the combined training and validation data. The test set remained untouched during model selection and hyperparameter tuning.

In [ ]:
# Combine Train and Validation sets
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

# Sort chronologically
train_val_df = train_val_df.sort_values(['date', 'store']).reset_index(drop=True)

X_train_val = train_val_df.drop(columns=['date', 'weekly_sales'])

y_train_val = train_val_df['weekly_sales']

print("Train + Validation shape:", X_train_val.shape)
print("From:", train_val_df['date'].min())
print("To:", train_val_df['date'].max())

In [ ]:
final_knn = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor(
        n_neighbors=3,
        weights='distance',
        p=1
    ))
])

final_knn.fit(X_train_val, y_train_val)

# Final Test Evaluation

In [ ]:
final_test_results = model_evaluation(final_knn, 'Final Test', X_test, y_test)

In [ ]:
Distribution_Plot(
    final_knn,
    'Final Test',
    X_test,
    y_test,
    'Final KNN Model: Actual vs Predicted Sales on Test Data'
)

In [ ]:
print("Test rows:", len(test_df))
print("Test period:")
print("From:", test_df['date'].min())
print("To:", test_df['date'].max())

In [ ]:
y_test_pred = final_knn.predict(X_test)

plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_test_pred, alpha=0.6)

min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle='--')

plt.xlabel('Actual Weekly Sales ($)')
plt.ylabel('Predicted Weekly Sales ($)')
plt.title('Actual vs Predicted Weekly Sales - Final KNN Model')

plt.tight_layout()
plt.show()

**Error Analysis**

In [ ]:
error_analysis = test_df[
    ['date', 'store', 'weekly_sales']
].copy()

error_analysis['predicted_sales'] = y_test_pred

error_analysis['error'] = (
    error_analysis['weekly_sales']
    - error_analysis['predicted_sales']
)

error_analysis['absolute_error'] = (
    error_analysis['error'].abs()
)

error_analysis = error_analysis.sort_values(
    'absolute_error',
    ascending=False
)

error_analysis.head(10)

In [ ]:
train_val_stores = set(X_train_val['store'].unique())
test_stores = set(X_test['store'].unique())

unseen_test_stores = sorted(test_stores - train_val_stores)

print("Stores in Train + Validation:", len(train_val_stores))
print("Stores in Test:", len(test_stores))
print("Unseen stores in Test:", unseen_test_stores)

In [ ]:
unseen_mask = X_test['store'].isin(unseen_test_stores)

print("Unseen-store test rows:", unseen_mask.sum())
print("Seen-store test rows:", (~unseen_mask).sum())

print("\nUnseen Stores Performance:")
model_evaluation(
    final_knn,
    'Unseen Stores',
    X_test[unseen_mask],
    y_test[unseen_mask]
)

print("\nSeen Stores Performance:")
model_evaluation(
    final_knn,
    'Seen Stores',
    X_test[~unseen_mask],
    y_test[~unseen_mask]
)

## Final Results, Conclusion, and Limitations

### Final Model Selection

Several regression models were trained and evaluated using a chronological train-validation-test strategy. The models were compared on the validation set before accessing the test set.

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| KNN (Tuned) | 107,001.87 | 180,165.53 | 0.9208 |
| Random Forest (Default) | 116,053.45 | 181,645.85 | 0.9195 |
| Random Forest (Tuned) | 127,894.47 | 197,748.98 | 0.9046 |
| Polynomial + Ridge (Tuned) | 130,151.61 | 203,711.80 | 0.8988 |
| Decision Tree (Tuned) | 135,817.54 | 214,637.13 | 0.8876 |
| Linear Regression | 193,553.10 | 281,574.55 | 0.8066 |

The tuned K-Nearest Neighbors (KNN) model achieved the lowest validation RMSE and was therefore selected as the final model.

The selected hyperparameters were:

- `n_neighbors = 3`
- `weights = 'distance'`
- `p = 1`

After model selection, the final KNN pipeline was retrained using the combined training and validation data and evaluated once on the held-out test set.

---

### Final Test Results

The final model was evaluated on 1,182 observations covering the period from April 13, 2012 to October 26, 2012.

| Metric | Final Test Result |
|---|---:|
| MAE | $72,030.09 |
| RMSE | $124,586.80 |
| R² | 0.9435 |

The final KNN model achieved an R² of 0.9435, indicating that it explained approximately 94.35% of the variance in weekly sales within the held-out test set.

The MAE indicates an average absolute prediction error of approximately $72,030. The higher RMSE of approximately $124,587 indicates that some observations had substantially larger prediction errors.

The Actual vs. Predicted and distribution plots also showed that predicted sales generally followed the observed sales pattern, although larger prediction errors occurred for some observations.

---

### Error Analysis

Further error analysis showed that prediction performance differed substantially between stores that were represented in the training data and stores that were not.

| Test Group | Observations | MAE | RMSE | R² |
|---|---:|---:|---:|---:|
| Seen Stores | 1,131 | $60,637.24 | $94,567.81 | 0.9681 |
| Unseen Stores | 51 | $324,683.38 | $401,766.69 | -0.3566 |

Stores **12, 28, and 38** were present in the test set but absent from the combined training and validation data after outlier filtering.

The model performed strongly on stores represented during training, achieving an R² of 0.9681. In contrast, its performance deteriorated substantially for the unseen stores, where R² decreased to -0.3566.

Several of the largest individual prediction errors were associated with stores 28 and 38. This suggests that store identity is an important source of information for the KNN model and that predictions for previously unseen stores are considerably less reliable.

---

### Limitations

1. **Unseen store categories:**  
   Stores 12, 28, and 38 were not represented in the final training data but appeared in the test set. Although `OneHotEncoder(handle_unknown='ignore')` allowed the pipeline to process these stores without producing an error, the model had not learned store-specific information for them.

2. **Effect of outlier filtering:**  
   Outlier filtering was performed before model training and resulted in some stores being absent from the training and validation periods. Therefore, the final evaluation represents performance on the filtered dataset, and the preprocessing decision contributed to the unseen-store issue identified during error analysis.

3. **Limited generalization to new stores:**  
   The strong overall test performance should not be interpreted as evidence that the model can reliably predict sales for completely unseen store identities. The separate evaluation demonstrated substantially weaker performance for such stores.

4. **Large individual prediction errors:**  
   Although the overall evaluation metrics indicated good predictive performance, RMSE was considerably higher than MAE, indicating the presence of observations with large prediction errors. The error analysis confirmed several substantial underpredictions and overpredictions.

5. **Time coverage:**  
   The dataset covers a limited historical period. Therefore, the model's performance reflects the sales patterns and economic conditions represented within this dataset and should not automatically be assumed to remain unchanged in later periods.

---

### Conclusion

This project developed and compared multiple machine learning regression models for predicting Walmart weekly sales using a chronological evaluation strategy.

Among the evaluated models, the tuned KNN model achieved the lowest validation RMSE and was selected as the final model. After retraining on the combined training and validation data, it achieved a test MAE of $72,030.09, RMSE of $124,586.80, and R² of 0.9435.

The results demonstrate strong predictive performance for stores represented in the training data. However, the error analysis revealed an important limitation: performance decreased substantially for stores that were unseen during training.

Therefore, the final model demonstrated strong predictive performance on the held-out test period for stores represented in its training data, while predictions for previously unseen stores should be interpreted with caution.

## Model Saving and Verification

The final trained KNN pipeline, including all preprocessing steps and the regression model, was saved using Joblib for future use without requiring the model to be retrained.

In [ ]:
import joblib

joblib.dump(final_knn, 'walmart_sales_knn_pipeline.pkl')

print("Final model saved successfully.")

In [ ]:
loaded_model = joblib.load(
    'walmart_sales_knn_pipeline.pkl'
)

loaded_test_results = model_evaluation(
    loaded_model,
    'Reloaded Final Test',
    X_test,
    y_test
)

The reloaded model produced the same test metrics, confirming that the complete preprocessing and prediction pipeline was serialized successfully.